# Paint 10M FineWeb embeddings by closeness to whatever you just lassoed

The thing no other interactive embedding viewer can do in a notebook:

1. Lasso a cluster in the widget.
2. Run a cell that uses `w.selected_indices` to compute a per-point score.
3. Hand the score back to the widget — **every one of the 10M points
   re-shades in milliseconds** because it's one GPU buffer swap.

Below: centroid in the lassoed region → Gaussian falloff → recolor.

In [ ]:
import numpy as np
from pathlib import Path
from stipple import Stipple

DATA = Path.home() / 'Sites/Thinking/Projects/Research/Stipple/app/public/data'

# 10M rows × 10 f32 columns. Cols 0,1 are UMAP-projected positions of
# FineWeb-Edu embeddings; the visible map IS the embedding here.
arr = np.fromfile(DATA / 'fineweb-10m-gpu-chunk00.f32', dtype=np.float32).reshape(-1, 10)
xs = np.ascontiguousarray(arr[:, 0])
ys = np.ascontiguousarray(arr[:, 1])

print(f'{len(xs):,} FineWeb documents loaded')

## Render

**Try it:** scroll-wheel to zoom, then **shift+drag** around a cluster you'd like to be the seed.

In [ ]:
# Scatter mode (one disk per point) so update_color recolors each
# point directly — density mode bins by count and would ignore the
# per-point scores. Stipple scatters 10M @ 60 FPS on M4.
w = Stipple(x=xs, y=ys, color=np.zeros(len(xs)), color_kind='continuous')
w

## Paint by closeness to the lasso

After lassoing, re-run this cell. Each point gets a Gaussian score
from the lassoed centroid; the widget repaints all 10M points in one
GPU buffer swap.

In [ ]:
idx = w.selected_indices
if len(idx) == 0:
    print('No selection yet. Shift+drag in the widget above, then re-run this cell.')
else:
    seed_x = float(xs[idx].mean())
    seed_y = float(ys[idx].mean())
    sigma = 0.05
    sims = np.exp(-((xs - seed_x)**2 + (ys - seed_y)**2) / (2 * sigma**2))
    w.update_color(sims)
    print(f'Lassoed {len(idx):,} · centroid ({seed_x:.3f}, {seed_y:.3f}) · '
          f'10M points repainted in one GPU upload')